In [0]:
# -- implement SCD 2


# -- 1.create target table from source table (one time activity)
# -- 2.create data frame for source and target tables
# -- 3.create delta table object for target table. it requires to perform merge operations.
# -- 4.identify modified records from source table (based on key column like salary)
# -- 5.expire old record (like update is_current=False in target table)
# -- 6.insert modified records in target table
# --     make sure to set is_current=True for new records
# -- 7. Find new records in source table then add new records in target table
# --     make sure to set is_current=True for new records

from delta.tables import *
from pyspark.sql.types import *
from pyspark.sql.functions import col,lit

data1=[(1,"ravi",9000),(2,"kumar",3000),(3,"aarush",4000)]
tgt_data=[(1,"ravi",1000,"Yes"),(2,"kumar",3000,"Yes")]

# define source schema using struct type

src_schema=StructType([
    StructField("id",IntegerType(),True),
    StructField("employee_name",StringType(),True),
    StructField("salary",StringType(),True)
])

# define target schema using struct type
tgt_schema=StructType([
    StructField("id",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("salary",StringType(),True),
    StructField("is_current",StringType(),True)])

# create source and target data frames
src_df=spark.createDataFrame(data1,src_schema)
tgt_df=spark.createDataFrame(tgt_data,tgt_schema)

# create source and target table
# src_df.write.format("delta").mode("overwrite").saveAsTable("scd_source")
# tgt_df.write.format("delta").mode("overwrite").saveAsTable("scd_target")

# handle renamed column in source. here emp is changed to emp_name in source. it should be handled
rename_mapping={

    "employee_name":"name"
}

for old_col,new_col in rename_mapping.items():

    if old_col in src_df.columns:

        src_df=src_df.withColumnRenamed(
            old_col,
            new_col
        )

# manage change in data type in source columns and fix them dynamically
tgt_types = dict(tgt_df.dtypes)

# compare and cast source columns
for c, dtype in src_df.dtypes:

    if c in tgt_types and dtype != tgt_types[c]:

        src_df = src_df.withColumn(
            c,
            col(c).cast(tgt_types[c])
        )

# create delta object for target table (it is required to perform merge operations which is not possible with regular data frame)
delta_table_obj=DeltaTable.forName(spark,"scd_target")

# identify updated records in source table
change_src_df = (
    src_df.alias("src")
    .join(
        tgt_df.alias("tgt"),
        col("src.id") == col("tgt.id")
    )
    .filter(
        (col("src.salary") != col("tgt.salary")) &
        (col("tgt.is_current") == "Yes")
    )
    .select("src.id", "src.name", "src.salary"))

# expire updated record in target table, means set is_active=False
delta_table_obj.alias("tgt") \
.merge(
    change_src_df.alias("src"),
    (col("tgt.id") == col("src.id")) & (col("tgt.is_current") == "Yes")
) \
.whenMatchedUpdate(
    condition=
        (col("tgt.salary") != col("src.salary")),
    set={
        "is_current": lit("No")
    }
) \
.execute()


# insert updated records in target table.
updated_records_df=change_src_df.withColumn("is_current",lit("Yes"))

updated_records_df.write.format("delta").mode("append").saveAsTable("scd_target")


# insert new records which are not present in target before(means brand new records)
new_records_df=src_df.alias("src").join(tgt_df.alias("tgt"),col("src.id")==col("tgt.id"),"left_anti")
new_records_df=new_records_df.withColumn("is_current",lit("Yes"))
new_records_df.write.format("delta").mode("append").saveAsTable("scd_target")

new_records_df.printSchema()

spark.table("scd_target").show()







In [0]:
%sql
drop table scd_target

